In [1]:
import os
import warnings
from pymatgen.core import Structure
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.io.vasp.sets import MPRelaxSet

# 忽略一些不重要的 pymatgen 警告
warnings.filterwarnings("ignore")

# ================= 1. 路径配置 =================
# 输入文件路径 (确保这些文件存在！)
elec_path = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
anode_path = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"

# 输出目录
output_interface_dir = "Na_Na3SbS4_Data/Interface_structures"
output_aimd_dir = "Na_Na3SbS4_Data/Interface_AIMD_runs"

# 创建目录
os.makedirs(output_interface_dir, exist_ok=True)
os.makedirs(output_aimd_dir, exist_ok=True)

# 检查文件是否存在
if not os.path.exists(elec_path) or not os.path.exists(anode_path):
    print(f"❌ 错误: 找不到输入文件！请检查路径：\n  {elec_path}\n  {anode_path}")
else:
    print(f"✅ 文件检查通过。准备构建界面...")
    print(f"  电解质: {elec_path}")
    print(f"  阳极:   {anode_path}")

✅ 文件检查通过。准备构建界面...
  电解质: Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp
  阳极:   Na_Na3SbS4_Data/structures/Na_mp-127.vasp


In [6]:
# ================= 2. 替代方案：手动堆叠 (瘦身版 - 100原子) =================
import os
import numpy as np
from pymatgen.core import Structure, Lattice

def manual_stacking_small(elec_file, anode_file):
    print("✂️ 启动瘦身版堆叠 (目标 ~100 原子)...")
    
    if not os.path.exists(elec_file) or not os.path.exists(anode_file):
        print("❌ 找不到文件，请检查路径！")
        return []
        
    elyte = Structure.from_file(elec_file) # Na3SbS4
    anode = Structure.from_file(anode_file) # Na

    # --- 关键修改点：减小扩胞倍数 ---
    # Na3SbS4: 2x2x1 (16 * 4 = 64 原子)
    # 边长约 14.4 埃 (满足 >10埃 的 cutoff 要求)
    elyte.make_supercell([2, 2, 1])
    
    # Na: 3x3x2 (2 * 18 = 36 原子)
    # 边长约 12.7 埃 -> 将被强制拉伸到 14.4 埃
    # Z方向扩2倍保持一定厚度
    anode.make_supercell([3, 3, 2])
    
    print(f"  扩胞后原子数: Na3SbS4={len(elyte)}, Na={len(anode)}")
    print(f"  预计总原子数: {len(elyte) + len(anode)}")

    # 以电解质(Na3SbS4)的晶格为基准
    target_lattice_matrix = elyte.lattice.matrix.copy()
    # Z 轴总高度设为 30 埃 (比之前小一点，因为原子少了)
    target_lattice_matrix[2, 2] = 30.0 
    
    combined_structure = Structure(Lattice(target_lattice_matrix), [], [])

    # 1. 放入电解质
    for site in elyte:
        combined_structure.append(site.specie, site.coords, coords_are_cartesian=True)

    max_z_elyte = max([site.coords[2] for site in elyte])
    interface_gap = 2.5 
    z_offset = max_z_elyte + interface_gap

    # 2. 放入 Na (并拉伸适配)
    scale_x = elyte.lattice.a / anode.lattice.a
    scale_y = elyte.lattice.b / anode.lattice.b
    
    print(f"  ⚠️ 注意: Na 将在 XY 方向被拉伸 {scale_x:.2f} 倍 (这在高温MD中是可接受的)")

    for site in anode:
        new_coords = site.coords.copy()
        new_coords[0] *= scale_x
        new_coords[1] *= scale_y
        new_coords[2] += z_offset
        
        combined_structure.append(site.specie, new_coords, coords_are_cartesian=True)

    # 3. 微扰与保存
    combined_structure.perturb(0.1)
    
    # 存为不同的名字以示区分
    save_path = "Na_Na3SbS4_Data/Interface_structures/Interface_Small_100atoms.vasp"
    combined_structure.to(filename=save_path, fmt="poscar")
    
    print(f"✅ 瘦身完成！")
    print(f"  最终原子数: {len(combined_structure)}")
    print(f"  保存文件: {save_path}")
    
    return [save_path]

# --- 运行 ---
actual_elec_path = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
actual_anode_path = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"

# 运行这个新函数
saved_interfaces = manual_stacking_small(actual_elec_path, actual_anode_path)

✂️ 启动瘦身版堆叠 (目标 ~100 原子)...
  扩胞后原子数: Na3SbS4=32, Na=18
  预计总原子数: 50
  ⚠️ 注意: Na 将在 XY 方向被拉伸 1.14 倍 (这在高温MD中是可接受的)
✅ 瘦身完成！
  最终原子数: 50
  保存文件: Na_Na3SbS4_Data/Interface_structures/Interface_Small_100atoms.vasp


In [3]:
import os
import math
import warnings
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPRelaxSet

# ================= 配置区域 =================

# 1. 输入目录 (请确认这里是纯相结构目录，还是界面结构目录)
# 建议：如果是跑界面，改成 "Na_Na3SbS4_Data/Interface_structures"
input_dir = "Na_Na3SbS4_Data/Interface_structures" 

# 2. 输出目录
base_output_dir = "Na_Na3SbS4_Data/Interface_AIMD_runs"
os.makedirs(base_output_dir, exist_ok=True)

# 3. AIMD 参数设置 (针对 Na/Na3SbS4 界面优化)
aimd_settings = {
    # --- 运动控制 ---
    "IBRION": 0,          # MD 模式
    "NSW": 4000,          # 步数: 纯相建议 2000-4000; 界面建议 4000-8000 (跑久一点看反应)
    "POTIM": 2.0,         # 步长 2.0 fs (Na 很轻，如果报错尝试降为 1.5)
    "TEBEG": 800,         # 起始温度 800K (高温加速采样)
    "TEEND": 800,         # 结束温度
    "ISYM": 0,            # 关闭对称性 (必须)
    "SMASS": 0,           # Nose-Hoover 热浴
    
    # --- 晶胞与采样 ---
    "ISIF": 2,            # 固定体积 (NVT)。界面计算强烈建议固定体积，防止真空层塌陷。
    "KBLOCK": 1,          # [核心] 每 1 步保存一次 XDATCAR，最大化训练数据量
    
    # --- 电子步优化 (大体系/金属体系必备) ---
    "ALGO": "Fast",       # 电子步算法
    "PREC": "Normal",     # 精度
    "LREAL": "Auto",      # 投影算符自动 (加速大体系计算)
    "NELM": 100,          # 允许更多电子步迭代 (高温下电荷晃动，难收敛)
    "ISMEAR": 0,          # Gaussian Smearing
    "SIGMA": 0.1,         # [核心] 0.1 eV。因为有金属 Na，太小(0.05)会导致不收敛。
    
    # --- 输出精简 ---
    "LWAVE": False,       # 不存波函数
    "LCHARG": False,      # 不存电荷密度
    "NCORE": 16,          # 并行优化 (单节点64核推荐设为 8, 16 或 32)
}

# ================= 处理流程 =================
print(f"📂 读取结构目录: {input_dir} ...")

if not os.path.exists(input_dir):
    print(f"❌ 错误: 找不到目录 {input_dir}")
else:
    files = [f for f in os.listdir(input_dir) if f.endswith(('.cif', '.vasp'))]
    print(f"🔍 发现 {len(files)} 个结构文件。")

    for filename in files:
        file_path = os.path.join(input_dir, filename)
        struct_name = os.path.splitext(filename)[0]
        
        try:
            # 1. 加载结构
            structure = Structure.from_file(file_path)
            
            # 2. 自动扩胞 (Supercell) - 确保最小边长 > 10 Angstrom
            # 这一步对 DeepMD 至关重要，防止原子看见自己的镜像
            min_length = 10.0
            lengths = structure.lattice.abc
            scaling_matrix = [max(1, int(math.ceil(min_length / l))) for l in lengths]
            
            # 如果是界面结构(Interface)，通常本身就很大，可能 scaling_matrix 全是 1，这没问题
            structure.make_supercell(scaling_matrix)
            
            # 3. 创建任务文件夹
            task_dir = os.path.join(base_output_dir, struct_name)
            os.makedirs(task_dir, exist_ok=True)
            
            # 4. 生成 VASP 输入文件
            # 使用 MPRelaxSet 作为模板，但注入我们的 AIMD 设置
            vis = MPRelaxSet(
                structure, 
                user_incar_settings=aimd_settings,
                user_kpoints_settings={"reciprocal_density": 50}, # 稀疏 K 点 (Gamma only 附近)
                user_potcar_functional="PBE"
            )
            
            vis.write_input(task_dir)
            print(f"✅ 生成成功: {struct_name} (原子数: {structure.num_sites}) -> {task_dir}")
            
        except Exception as e:
            print(f"❌ 跳过 {filename}: {e}")
            if "POTCAR" in str(e):
                print("   (提示: 可能是服务器未配置 pymatgen 伪势库路径)")

    print(f"\n🎉 全部完成！输入文件已准备在 '{base_output_dir}'")

📂 读取结构目录: Na_Na3SbS4_Data/Interface_structures ...
🔍 发现 1 个结构文件。
✅ 生成成功: Interface_Small_100atoms (原子数: 50) -> Na_Na3SbS4_Data/Interface_AIMD_runs/Interface_Small_100atoms

🎉 全部完成！输入文件已准备在 'Na_Na3SbS4_Data/Interface_AIMD_runs'


In [4]:
import os
import glob
from dpdispatcher import Machine, Resources, Task, Submission

# 1. 设置机器环境 (不变)
machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

# 2. 设置资源 (修改时间与任务名)
resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g", 
    group_size=1,
    module_list=["vasp/6.3.0-intel-2021.4.0"], # 确保和你之前成功的一致
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --ntasks=64",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn",
        "#SBATCH --time=72:00:00",       # [修改] 100原子建议给足48小时
        "#SBATCH --job-name=Na_Interface" # [修改] 方便识别
    ]
)

# 3. 设置运行命令 (使用绝对路径最稳)
# 请替换为你之前确认过的真实 VASP 路径
vasp_exe = "/dssg/opt/icelake/linux-centos8-icelake/oneapi-2021.4.0/vasp/vasp.6.3.0/bin/vasp_std"

setup_env = (
    "ulimit -s unlimited && "
    "ulimit -l unlimited && "
    "export I_MPI_PMI_LIBRARY=/usr/lib64/libpmi.so && "
    "export I_MPI_FABRICS=shm:ofi && "
    "export I_MPI_PMI=pmi"
)
command = f"{setup_env} && mpirun -n 64 {vasp_exe}"

# 4. 搜集任务 (修改路径)
task_list = []
# [关键修改] 这里指向你刚刚生成 input 的文件夹
work_base = "Na_Na3SbS4_Data/Interface_AIMD_runs" 
search_pattern = os.path.join(work_base, "*")

print(f"📂 正在扫描任务目录: {work_base} ...")

for folder_path in glob.glob(search_pattern):
    if os.path.isdir(folder_path):
        folder_name = os.path.basename(folder_path)
        # 过滤掉非任务文件夹（如果有的话）
        if "Interface" not in folder_name: 
            continue
            
        print(f"  -> 发现任务: {folder_name}")
        
        task = Task(
            command=command,
            task_work_path=folder_name,
            forward_files=[], 
            backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR', 'XDATCAR'] 
        )
        task_list.append(task)

# 5. 提交
if len(task_list) > 0:
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )
    print(f"🚀 正在提交 {len(task_list)} 个界面任务...")
    submission.run_submission()
    print("✅ 提交完成！")
else:
    print("❌ 未找到任务，请检查路径是否正确。")

📂 正在扫描任务目录: Na_Na3SbS4_Data/Interface_AIMD_runs ...
  -> 发现任务: Interface_Small_100atoms
🚀 正在提交 1 个界面任务...
2025-12-09 16:14:55,368 - INFO : info:check_all_finished: False
2025-12-09 16:14:55,456 - INFO : job: 33c0d71be62f35390acbfc7fc40c94c4eed17322 submit; job_id is 50726189


In [1]:
import os
import warnings
from pymatgen.core import Structure
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.io.vasp.sets import MPRelaxSet

# 忽略一些不重要的 pymatgen 警告
warnings.filterwarnings("ignore")

# ================= 1. 路径配置 =================
# 输入文件路径 (确保这些文件存在！)
elec_path = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
anode_path = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"

# 输出目录
output_interface_dir = "Na_Na3SbS4_Data/Interface_structures"
output_aimd_dir = "Na_Na3SbS4_Data/Interface_AIMD_runs"

# 创建目录
os.makedirs(output_interface_dir, exist_ok=True)
os.makedirs(output_aimd_dir, exist_ok=True)

# 检查文件是否存在
if not os.path.exists(elec_path) or not os.path.exists(anode_path):
    print(f"❌ 错误: 找不到输入文件！请检查路径：\n  {elec_path}\n  {anode_path}")
else:
    print(f"✅ 文件检查通过。准备构建界面...")
    print(f"  电解质: {elec_path}")
    print(f"  阳极:   {anode_path}")

✅ 文件检查通过。准备构建界面...
  电解质: Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp
  阳极:   Na_Na3SbS4_Data/structures/Na_mp-127.vasp


In [7]:
import os
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.core.surface import SlabGenerator
from pymatgen.analysis.interfaces.zsl import ZSLGenerator

def force_build_interface_fixed(elec_file, anode_file):
    print("☢️ 启动强制堆叠模式 (修复版 V2)...")

    if not os.path.exists(elec_file) or not os.path.exists(anode_file):
        print("❌ 文件未找到")
        return

    # 1. 准备基础切片
    elyte_bulk = Structure.from_file(elec_file).get_primitive_structure()
    anode_bulk = Structure.from_file(anode_file).get_primitive_structure()
    
    # 切 (001) 面
    slab_elyte = SlabGenerator(elyte_bulk, (0,0,1), min_slab_size=8, min_vacuum_size=10, center_slab=True).get_slab()
    slab_anode = SlabGenerator(anode_bulk, (0,0,1), min_slab_size=8, min_vacuum_size=10, center_slab=True).get_slab()

    print(f"  - 原始切片原子数: Electrolyte={len(slab_elyte)}, Anode={len(slab_anode)}")

    # 2. 运行 ZSL 算法
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.2,  # 20% 面积容差
        max_area=400,            
        max_length_tol=0.2,      # 20% 边长容差
        max_angle_tol=0.1        
    )
    
    # 提取 Lattice 传入
    # 注意: 最新版 ZSL 建议直接传 matrix 或 lattice，这里我们为了兼容性传入 basis vectors
    u_vec = slab_elyte.lattice.matrix[0:2, 0:2]
    v_vec = slab_anode.lattice.matrix[0:2, 0:2]

    print("  - 正在计算匹配矩阵...")
    matches = list(zsl(u_vec, v_vec))
    
    if not matches:
        print("❌ 依然未找到匹配。将使用默认 1x1 强行堆叠。")
        # 手动造一个 Dummy Match 对象
        class DummyMatch:
            substrate_transformation = [[1, 0], [0, 1]]
            film_transformation = [[2, 0], [0, 2]] # 假设 Anode 扩两倍
        matches = [DummyMatch()]

    # 3. 挑选最佳匹配 (兼容性修复部分)
    best_match_data = None
    best_atom_count_diff = 9999
    
    print(f"  - 找到 {len(matches)} 个潜在匹配方案，正在筛选...")

    for match in matches:
        # --- 兼容性修复 START ---
        # 尝试从 match 对象中提取矩阵，兼容不同版本的 Pymatgen
        sub_mat = None
        film_mat = None
        
        # 检查是否是字典 (旧版)
        if isinstance(match, dict):
            sub_mat = match.get('sub_matrix')
            film_mat = match.get('film_matrix')
        else:
            # 尝试对象属性 (新版)
            # 属性名可能是 substrate_transformation 或 sub_matrix
            if hasattr(match, "substrate_transformation"):
                sub_mat = match.substrate_transformation
                film_mat = match.film_transformation
            elif hasattr(match, "sub_matrix"):
                sub_mat = match.sub_matrix
                film_mat = match.film_matrix
        
        if sub_mat is None or film_mat is None:
            print("  ⚠️ 警告: 无法识别的 ZSLMatch 格式，跳过此方案")
            continue
        
        # 转换为 numpy 数组以计算行列式
        sub_mat = np.array(sub_mat)
        film_mat = np.array(film_mat)
        # --- 兼容性修复 END ---

        # 估算原子数
        scale_elyte = abs(np.linalg.det(sub_mat))
        scale_anode = abs(np.linalg.det(film_mat))
        total_atoms = (len(slab_elyte) * scale_elyte) + (len(slab_anode) * scale_anode)
        
        diff = abs(total_atoms - 150)
        
        # 记录最佳方案的数据
        if diff < best_atom_count_diff:
            best_atom_count_diff = diff
            best_match_data = {
                'sub_matrix': sub_mat,
                'film_matrix': film_mat,
                'total_atoms': total_atoms
            }

    if best_match_data is None:
        print("❌ 无法解析任何匹配结果。")
        return

    print(f"  - 选中方案: Elyte x{best_match_data['sub_matrix'].tolist()} | Anode x{best_match_data['film_matrix'].tolist()}")
    print(f"  - 预计原子数: {int(best_match_data['total_atoms'])}")

    # 4. 执行扩胞
    # 构造 3x3 扩胞矩阵 (Z轴保持为 1)
    m_elyte = np.eye(3)
    m_elyte[0:2, 0:2] = best_match_data['sub_matrix']
    
    m_anode = np.eye(3)
    m_anode[0:2, 0:2] = best_match_data['film_matrix']
    
    super_elyte = slab_elyte.copy()
    super_elyte.make_supercell(m_elyte)
    
    super_anode = slab_anode.copy()
    super_anode.make_supercell(m_anode)
    
    # 5. 强制堆叠逻辑 (同前)
    # 获取 Z 厚度
    anode_sites_z = [s.coords[2] for s in super_anode]
    anode_thickness = max(anode_sites_z) - min(anode_sites_z)
    
    elyte_sites_z = [s.coords[2] for s in super_elyte]
    elyte_thickness = max(elyte_sites_z) - min(elyte_sites_z)
    
    # 计算新的 C 轴
    total_c = elyte_thickness + anode_thickness + 2.5
    
    # 建立新晶格 (以 Elyte 为基准)
    target_lattice = super_elyte.lattice
    new_matrix = target_lattice.matrix.copy()
    new_matrix[2, 0] = 0
    new_matrix[2, 1] = 0
    new_matrix[2, 2] = total_c
    
    final_struct = Structure(Lattice(new_matrix), [], [])
    
    # 5.1 放入 Elyte (底部)
    min_elyte_z = min(elyte_sites_z)
    for site in super_elyte:
        new_coords = list(site.coords)
        new_coords[2] -= min_elyte_z 
        new_coords[2] += 1.0 # 底部缓冲
        final_struct.append(site.specie, new_coords, coords_are_cartesian=True)
        
    # 5.2 放入 Anode (顶部) - 强制拉伸 XY
    current_anode_z_top = 1.0 + elyte_thickness + 2.5
    min_anode_z = min(anode_sites_z)
    
    # 这里的关键是：我们使用 Anode 的分数坐标 (Fractional)，
    # 但应用到 Elyte 的晶格矢量 (Lattice Vectors) 上。
    # 这会自动完成拉伸。
    
    vec_a = new_matrix[0]
    vec_b = new_matrix[1]
    
    for site in super_anode:
        u, v = site.frac_coords[0], site.frac_coords[1]
        
        # 计算 Z (Cartesian)
        old_z = site.coords[2]
        new_z_cart = (old_z - min_anode_z) + current_anode_z_top
        
        # 合成新坐标
        new_pos = (u * vec_a) + (v * vec_b)
        new_pos[2] = new_z_cart
        
        final_struct.append(site.specie, new_pos, coords_are_cartesian=True)

    # 6. 保存
    final_struct.perturb(0.1)
    save_path = f"Na_Na3SbS4_Data/Interface_structures/Interface_Fixed_{len(final_struct)}atoms.vasp"
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    final_struct.to(filename=save_path, fmt="poscar")
    
    print(f"✅ 修复版堆叠完成！")
    print(f"  - 最终原子数: {len(final_struct)}")
    print(f"  - 保存路径: {save_path}")

# --- 运行 ---
actual_elec_path = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
actual_anode_path = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"

force_build_interface_fixed(actual_elec_path, actual_anode_path)

☢️ 启动强制堆叠模式 (修复版 V2)...
  - 原始切片原子数: Electrolyte=16, Anode=3
  - 正在计算匹配矩阵...
  - 找到 14572 个潜在匹配方案，正在筛选...
  - 选中方案: Elyte x[[1.0, 2.0], [0.0, 9.0]] | Anode x[[1.0, 0.0], [0.0, 3.0]]
  - 预计原子数: 153
✅ 修复版堆叠完成！
  - 最终原子数: 153
  - 保存路径: Na_Na3SbS4_Data/Interface_structures/Interface_Fixed_153atoms.vasp


In [9]:
import os
import math
import warnings
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.io.vasp.inputs import Kpoints

# ================= 配置区域 =================

input_dir = "Na_Na3SbS4_Data/Interface_structures"
base_output_dir = "Na_Na3SbS4_Data/Interface_AIMD_runs"
os.makedirs(base_output_dir, exist_ok=True)

# AIMD 参数设置 (保持你的设置不变)
aimd_settings = {
    "IBRION": 0,          
    "NSW": 5000,          
    "POTIM": 2.0,         
    "TEBEG": 800,         
    "TEEND": 800,
    "ISYM": 0,            
    "SMASS": 0,           
    "ISIF": 2,            
    "KBLOCK": 1,          
    "ALGO": "Fast",       
    "PREC": "Normal",     
    "LREAL": "Auto",      
    "NELM": 100,          
    "ISMEAR": -1,         # Fermi Smearing (适合金属界面)
    "SIGMA": 0.1,         
    "ISPIN": 1,           
    "LWAVE": False,       
    "LCHARG": False,      
    "NCORE": 8,           
}

# ================= 处理流程 =================
print(f"📂 读取结构目录: {input_dir} ...")

if not os.path.exists(input_dir):
    print(f"❌ 错误: 找不到目录 {input_dir}")
else:
    files = [f for f in os.listdir(input_dir) if f.endswith(('.cif', '.vasp'))]
    files.sort() 
    print(f"🔍 发现 {len(files)} 个结构文件。")

    for filename in files:
        file_path = os.path.join(input_dir, filename)
        struct_name = os.path.splitext(filename)[0]
        
        try:
            # 1. 加载结构
            structure = Structure.from_file(file_path)
            
            # 2. 扩胞检查
            min_length = 10.0
            lengths = structure.lattice.abc
            scaling_matrix = [max(1, int(math.ceil(min_length / l))) for l in lengths]
            
            if any(x > 1 for x in scaling_matrix):
                print(f"  - {struct_name}: 执行扩胞 {scaling_matrix}")
                structure.make_supercell(scaling_matrix)
            
            # 3. 创建目录
            task_dir = os.path.join(base_output_dir, struct_name)
            os.makedirs(task_dir, exist_ok=True)
            
            # 4. 生成 VASP 输入文件 (修复点在这里!)
            # 我们先创建 Kpoints 对象
            gamma_only = Kpoints.gamma_automatic()
            
            # 然后直接传给 MPRelaxSet
            vis = MPRelaxSet(
                structure, 
                user_incar_settings=aimd_settings,
                user_potcar_functional="PBE",
                user_kpoints_settings=gamma_only  # <--- 直接在这里传入对象
            )
            
            # 写入文件
            vis.write_input(task_dir)
            print(f"✅ 生成成功: {struct_name} (原子数: {structure.num_sites}) -> {task_dir}")
            
        except Exception as e:
            print(f"❌ 跳过 {filename}: {e}")

    print(f"\n🎉 全部完成！请检查 '{base_output_dir}'")

📂 读取结构目录: Na_Na3SbS4_Data/Interface_structures ...
🔍 发现 2 个结构文件。
✅ 生成成功: Interface_Fixed_153atoms (原子数: 153) -> Na_Na3SbS4_Data/Interface_AIMD_runs/Interface_Fixed_153atoms
✅ 生成成功: Interface_Small_100atoms (原子数: 50) -> Na_Na3SbS4_Data/Interface_AIMD_runs/Interface_Small_100atoms

🎉 全部完成！请检查 'Na_Na3SbS4_Data/Interface_AIMD_runs'


In [1]:
import os
from dpdispatcher import Machine, Resources, Task, Submission

# ================= 配置区域 =================

# 1. 基础路径
work_base = "Na_Na3SbS4_Data/Interface_AIMD_runs" 

# 2. 指定你要跑的那个文件夹名字 (精准打击)
target_folder_name = "Interface_Fixed_153atoms"

# 3. VASP 路径
vasp_exe = "/dssg/opt/icelake/linux-centos8-icelake/oneapi-2021.4.0/vasp/vasp.6.3.0/bin/vasp_std"

# ================= 机器与资源 =================

machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g", 
    group_size=1,
    module_list=["vasp/6.3.0-intel-2021.4.0"], 
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --ntasks=64",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn",
        "#SBATCH --time=72:00:00",      
        f"#SBATCH --job-name={target_folder_name}" # 任务名直接用文件夹名
    ]
)

setup_env = (
    "ulimit -s unlimited && "
    "ulimit -l unlimited && "
    "export I_MPI_PMI_LIBRARY=/usr/lib64/libpmi.so && "
    "export I_MPI_FABRICS=shm:ofi && "
    "export I_MPI_PMI=pmi"
)
command = f"{setup_env} && mpirun -n 64 {vasp_exe}"

# ================= 构建任务 =================

task_list = []
full_path = os.path.join(work_base, target_folder_name)

print(f"🎯 正在定位目标任务: {full_path} ...")

# 检查文件夹是否存在，并且里面有 INCAR
if os.path.isdir(full_path) and os.path.exists(os.path.join(full_path, "INCAR")):
    print(f"  ✅ 找到任务文件夹，准备提交...")
    
    task = Task(
        command=command,
        task_work_path=target_folder_name, # 注意：这里填相对路径(文件夹名)即可，因为 Submission 指定了 work_base
        forward_files=['INCAR', 'POSCAR', 'POTCAR', 'KPOINTS'], 
        backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR', 'XDATCAR', 'CONTCAR'] 
    )
    task_list.append(task)
else:
    print(f"❌ 错误: 找不到文件夹 {full_path} 或其中缺少 INCAR 文件！")

# ================= 提交执行 =================

if len(task_list) > 0:
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )
    
    submission.run_submission()
    print(f"🚀 任务 {target_folder_name} 已提交！")
    print("📋 请使用 'squeue' 查看状态。")
else:
    print("❌ 提交终止。")

🎯 正在定位目标任务: Na_Na3SbS4_Data/Interface_AIMD_runs/Interface_Fixed_153atoms ...
  ✅ 找到任务文件夹，准备提交...
2025-12-12 13:46:38,566 - INFO : info:check_all_finished: False
2025-12-12 13:46:38,711 - INFO : job: bf804ae8e041dfd6618d2970d416c1c4286ce2ed submit; job_id is 50793897
